# Day 06：Python 函数进阶与函数式管道 —— 重构智枢平台处理引擎

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第 6 天！我是你的主讲兼 Tech Lead 张工。前 5 天我们写出了环境检测、模板拼接和 JSON 解析，但代码大多平铺在脚本里。今天我们要做一次真正的**工程重构**：掌握 Python 函数的全套参数体系（位置参数、关键字参数、默认参数、`*args`、`**kwargs`），用**高阶函数 + `lambda`** 打造可插拔的 Prompt 清洗流水线，再用**递归**优雅地遍历任意深度的嵌套 JSON 对话树！

## 1. 第一步：函数参数体系与 `build_api_payload`
在调用大模型 API 时，有些参数是必选的（比如 `model`），有些是数量不固定的（比如任意多条 `messages`），有些有默认值（比如 `temperature=0.7`），还有些是厂商特有的扩展参数（比如 `max_tokens`、`stream`、`top_p`）。
Python 提供了完美匹配这一场景的参数顺序规范：
`def func(位置参数, *args, 默认关键字参数=..., **kwargs)`

In [1]:
import json


def build_api_payload(
    model: str,
    *messages: dict,
    temperature: float = 0.7,
    **extra_params,
) -> dict:
    """动态构造大模型 API 请求体字典。"""
    # 1. 校验必选位置参数 model 是否为空
    if not isinstance(model, str) or not model.strip():
        raise ValueError("model 必须是非空字符串")

    # 2. 校验默认关键字参数 temperature 范围（注意：它在 *messages 之后，天然成为 Keyword-Only 参数）
    if not (0.0 <= temperature <= 2.0):
        raise ValueError(f"temperature 必须在 [0.0, 2.0] 范围内，当前为: {temperature}")

    # 3. 校验 *args 收集到的每条消息是否为 dict
    for idx, msg in enumerate(messages):
        if not isinstance(msg, dict):
            raise TypeError(f"messages 第 {idx} 项必须是 dict 类型")

    # 4. 组装基础请求体，并将 **extra_params 收集的扩展参数合并进去
    payload = {
        "model": model.strip(),
        "messages": list(messages),  # *messages 是元组，转为 list 以符合 JSON 数组规范
        "temperature": temperature,
    }
    payload.update(extra_params)
    return payload


# 演示 1：仅传必选参数 + 两条消息（使用默认 temperature=0.7）
p1 = build_api_payload(
    "deepseek-chat",
    {"role": "system", "content": "你是智枢助手"},
    {"role": "user", "content": "请介绍 Python 函数"},
)
print("演示 1（默认参数）:")
print(json.dumps(p1, ensure_ascii=False, indent=2))

# 演示 2：覆盖 temperature 并通过 **kwargs 透传扩展参数
p2 = build_api_payload(
    "qwen-plus",
    {"role": "user", "content": "输出 Q3 财报摘要"},
    temperature=0.2,
    max_tokens=2048,
    stream=True,
)
print("\n演示 2（自定义温度 + **kwargs 扩展参数）:")
print(json.dumps(p2, ensure_ascii=False, indent=2))

演示 1（默认参数）:
{
  "model": "deepseek-chat",
  "messages": [
    {
      "role": "system",
      "content": "你是智枢助手"
    },
    {
      "role": "user",
      "content": "请介绍 Python 函数"
    }
  ],
  "temperature": 0.7
}

演示 2（自定义温度 + **kwargs 扩展参数）:
{
  "model": "qwen-plus",
  "messages": [
    {
      "role": "user",
      "content": "输出 Q3 财报摘要"
    }
  ],
  "temperature": 0.2,
  "max_tokens": 2048,
  "stream": true
}


## 2. 第二步：高阶函数与 `lambda` 匿名函数 —— `apply_pipeline`
在 Python 中，**函数是一等公民**，可以像普通变量一样作为参数传递给另一个函数（即**高阶函数**）。
搭配小巧的 `lambda` 匿名函数，我们可以把 Prompt 清洗逻辑拆解成一条“流水线”：想去空格就加一步，想脱敏就再插一步，完全不用修改主函数！

In [2]:
def apply_pipeline(text: str, *transforms) -> str:
    """高阶函数：按顺序将任意多个文本变换函数依次应用到 text 上。"""
    # 1. 类型防御：确保输入是字符串
    if not isinstance(text, str):
        raise TypeError("text 必须是 str 类型")

    result = text
    # 2. 遍历 *transforms 中的每个函数，依次处理并更新 result
    for idx, fn in enumerate(transforms):
        if not callable(fn):
            raise TypeError(f"transforms 第 {idx} 项不可调用: {fn!r}")
        result = fn(result)

    return result


# 模拟一条脏乱且含敏感词的用户原始 Prompt
raw_prompt = "   请帮我分析   【机密】星辰数科 Q3 营收数据   并给出建议！  "

# 将 str.strip 与两个 lambda 匿名函数串联成清洗管道
cleaned_prompt = apply_pipeline(
    raw_prompt,
    str.strip,                                       # 步骤 1：去除首尾空白
    lambda s: " ".join(s.split()),                   # 步骤 2：将中间连续空格压缩为单个空格
    lambda s: s.replace("【机密】", "[已脱敏]"),      # 步骤 3：敏感标签脱敏替换
)

print("清洗前:", repr(raw_prompt))
print("清洗后:", repr(cleaned_prompt))

清洗前: '   请帮我分析   【机密】星辰数科 Q3 营收数据   并给出建议！  '
清洗后: '请帮我分析 [已脱敏]星辰数科 Q3 营收数据 并给出建议！'


## 3. 第三步：`lambda` 高频实战 —— 大模型按价格与性价比排序
在企业级路由网关中，我们经常需要根据模型单价动态排序，优先选择性价比最高的模型。`sorted(iterable, key=lambda ...)` 是每个 Python 工程师每天都会用到的经典高阶函数组合。

In [3]:
# 智枢平台接入的模型定价清单（单位：元 / 百万 Token）
models = [
    {"name": "deepseek-chat", "input_price": 1.0, "output_price": 2.0},
    {"name": "qwen-max", "input_price": 20.0, "output_price": 60.0},
    {"name": "glm-4-flash", "input_price": 0.1, "output_price": 0.1},
    {"name": "gpt-4o-mini", "input_price": 1.1, "output_price": 4.4},
]

# 按输出单价 output_price 从低到高排序
sorted_by_output = sorted(models, key=lambda m: m["output_price"])
print("按输出单价从低到高排序:")
for m in sorted_by_output:
    print(f"  - {m['name']:15s} | 输入: ¥{m['input_price']:4.1f} | 输出: ¥{m['output_price']:4.1f}")

按输出单价从低到高排序:
  - glm-4-flash     | 输入: ¥ 0.1 | 输出: ¥ 0.1
  - deepseek-chat   | 输入: ¥ 1.0 | 输出: ¥ 2.0
  - gpt-4o-mini     | 输入: ¥ 1.1 | 输出: ¥ 4.4
  - qwen-max        | 输入: ¥20.0 | 输出: ¥60.0


## 4. 第四步：递归函数与嵌套 JSON 树遍历 —— `count_nested_tokens`
大模型 API 请求体与响应体往往是**多层嵌套的字典和列表**（比如 `payload -> messages -> list -> dict -> content`）。
面对深度未知的树状结构，手写多层 `for` 循环不仅丑陋而且容易漏层。**递归（Recursion）**只需要写清楚两个条件就能迎刃而解：
1. **基准条件（Base Case）**：遇到 `str` 时直接返回 `len(node)`；
2. **递归步骤（Recursive Step）**：遇到 `list` 遍历元素递归求和；遇到 `dict` 遍历 `values()` 递归求和；其他标量（`int`/`bool`/`None`）返回 `0`。

In [4]:
def count_nested_tokens(node) -> int:
    """递归遍历嵌套 JSON / 消息树，统计所有文本节点的字符数（估算 Token）。"""
    # 基准条件：叶子节点是字符串，直接返回字符长度
    if isinstance(node, str):
        return len(node)

    # 递归分支 1：列表节点，对每个子元素递归求和
    if isinstance(node, list):
        return sum(count_nested_tokens(item) for item in node)

    # 递归分支 2：字典节点，对每个 value 递归求和
    if isinstance(node, dict):
        return sum(count_nested_tokens(value) for value in node.values())

    # 其他标量类型（int, float, bool, None）不计入文本 Token
    return 0


# 综合串联：对前面构造的 p2 请求体进行嵌套 Token 估算
total_chars = count_nested_tokens(p2)
print("请求体 p2 的嵌套文本总字符数（估算 Token）:", total_chars)

请求体 p2 的嵌套文本总字符数（估算 Token）: 23
